# 02 — LLM Sentiment Scoring

**Goal:** Use a HuggingFace transformer to re-score every news headline with a financial-grade sentiment probability, then aggregate to a daily sentiment signal that the LSTM will consume.

**Model choice:** `ProsusAI/finbert` — a BERT model fine-tuned on financial text. Falls back to `distilbert-base-uncased-finetuned-sst-2-english` if FinBERT is unavailable.


## 2.1 Imports & load interim data


In [ ]:
import os
from pathlib import Path
import pandas as pd
import numpy as np
from tqdm.auto import tqdm

ROOT = Path.cwd()
for _ in range(6):
    if (ROOT / 'Data' / 'cryptonews.csv').exists() or (ROOT / 'notebooks').exists():
        break
    if ROOT == ROOT.parent:
        break
    ROOT = ROOT.parent
INTERIM = ROOT / 'notebooks' / 'interim'
INTERIM.mkdir(parents=True, exist_ok=True)

merged = pd.read_parquet(INTERIM / 'merged_daily.parquet')
print(f'Loaded {len(merged):,} daily rows')

## 2.2 Re-fetch raw headlines for LLM scoring
Daily aggregates lose headline-level signal. We re-pull the raw news CSV (cached locally after Notebook 01) and score each headline individually.


In [ ]:
LOCAL_NEWS = ROOT / 'Data' / 'cryptonews.csv'
news = pd.read_csv(LOCAL_NEWS)
# Re-apply Bug Fix 2
news['date'] = pd.to_datetime(news['date'], format='mixed', utc=True, errors='coerce')
news = news.dropna(subset=['date']).sort_values('date').reset_index(drop=True)
# Use the title as the primary text for sentiment scoring
news['text'] = news['title'].fillna('') + '. ' + news['text'].fillna('')
print(f'{len(news):,} headlines ready for LLM scoring')
news[['date', 'title']].head(3)

## 2.3 Load the HuggingFace sentiment pipeline


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline

DEVICE = 0 if torch.cuda.is_available() else -1
print(f'Torch device: {"cuda" if DEVICE == 0 else "cpu"}')

MODEL_CANDIDATES = [
    'ProsusAI/finbert',                                    # finance-tuned BERT
    'distilbert-base-uncased-finetuned-sst-2-english',    # fallback: general 2-class
]

pipe = None
for name in MODEL_CANDIDATES:
    try:
        print(f'Loading {name} ...')
        tok = AutoTokenizer.from_pretrained(name)
        mdl = AutoModelForSequenceClassification.from_pretrained(name)
        pipe = pipeline('sentiment-analysis', model=mdl, tokenizer=tok, device=DEVICE, truncation=True, max_length=512)
        print(f'  -> loaded {name}')
        break
    except Exception as e:
        print(f'  ! {name} failed: {e}')

assert pipe is not None, 'No sentiment model could be loaded.'

## 2.4 Score headlines in batches
We process headlines in batches of 64 for throughput. On CPU, the full 31k-row dataset takes ~15 minutes; on GPU it's under 2 minutes.


In [ ]:
BATCH = 64
texts = news['text'].astype(str).tolist()
scores = []
for i in tqdm(range(0, len(texts), BATCH), desc='LLM sentiment'):
    batch = texts[i:i+BATCH]
    try:
        results = pipe(batch)
    except Exception:
        # Fallback: score one-by-one to skip individual failures
        results = [pipe(t) if len(t) > 1 else {'label': 'neutral', 'score': 0.5} for t in batch]
    for r in results:
        label = r['label'].lower()
        prob = float(r['score'])
        if 'pos' in label:
            sentiment_score = prob
        elif 'neg' in label:
            sentiment_score = -prob
        else:
            sentiment_score = 0.0
        scores.append(sentiment_score)

news['llm_sentiment'] = scores
print(f'Mean LLM sentiment: {news.llm_sentiment.mean():.4f}')
print(f'Std               : {news.llm_sentiment.std():.4f}')
news[['date', 'title', 'llm_sentiment']].head(5)

## 2.5 Aggregate to daily LLM sentiment


In [ ]:
news['date_day'] = news['date'].dt.tz_convert(None).dt.floor('D')
daily_llm = (
    news.groupby('date_day')
         .agg(llm_sentiment_mean=('llm_sentiment', 'mean'),
              llm_sentiment_std=('llm_sentiment', 'std'),
              llm_headline_count=('llm_sentiment', 'size'),
              llm_pos_share=('llm_sentiment', lambda s: (s > 0.3).mean()),
              llm_neg_share=('llm_sentiment', lambda s: (s < -0.3).mean()))
         .reset_index()
         .rename(columns={'date_day': 'date'})
         .fillna({'llm_sentiment_std': 0})
)
print(f'{len(daily_llm):,} daily LLM sentiment rows')
daily_llm.head(3)

## 2.6 Merge LLM features into the daily dataset


In [ ]:
merged = merged.merge(daily_llm, on='date', how='left')
llm_cols = ['llm_sentiment_mean', 'llm_sentiment_std', 'llm_headline_count', 'llm_pos_share', 'llm_neg_share']
merged[llm_cols] = merged[llm_cols].fillna(0)
print(f'Final merged shape: {merged.shape}')
merged[['date', 'close', 'llm_sentiment_mean', 'llm_headline_count']].tail(5)

## 2.7 Persist for Notebook 03


In [ ]:
out_path = INTERIM / 'merged_with_llm_sentiment.parquet'
merged.to_parquet(out_path, index=False)
print(f'Wrote {len(merged):,} rows → {out_path}')

## 2.8 Summary
- Loaded a HuggingFace transformer (FinBERT with fallback to DistilBERT).
- Scored all 31k headlines in batches of 64.
- Aggregated to daily LLM sentiment features (mean, std, headline count, pos/neg shares).
- Persisted to `notebooks/interim/merged_with_llm_sentiment.parquet`.
